# Ask a tool, but check the request first

Imagine an assistant that wants to read the refund policy. It proposes `lookup_policy(topic="refund")`. Our program checks the tool name and input, looks up a fixed note and returns it. If a proposed tool is `delete_files`, the program refuses. A suggested action is never permission to execute it.

Before running: will the lookup find a refund answer? Should a request to delete files be accepted? Read [the agent lesson](../../docs/07-agents-mcp.md). This is a small **simulation** of tool authorization; it does not start an MCP server or contact any provider.

In [ ]:
POLICIES = {'refund': 'Refund requests are reviewed within 10 days.', 'shipping': 'Standard shipping takes three days.'}
def lookup_policy(topic: str) -> str:
    if topic not in POLICIES:
        return 'No matching policy.'
    return POLICIES[topic]
def run_tool_call(name: str, arguments: dict) -> str:
    if name != 'lookup_policy' or set(arguments) != {'topic'}:
        raise ValueError('Tool or schema not permitted')
    topic = arguments['topic']
    if not isinstance(topic, str) or len(topic) > 32:
        raise ValueError('Invalid topic')
    return lookup_policy(topic)
assert '10 days' in run_tool_call('lookup_policy', {'topic': 'refund'})
for name, args in [('delete_files', {}), ('lookup_policy', {'topic': '../secrets'}), ('lookup_policy', {'topic': 'a' * 100})]:
    try:
        result = run_tool_call(name, args)
        assert result == 'No matching policy.'
    except ValueError:
        pass
print(run_tool_call('lookup_policy', {'topic': 'shipping'}))

## Check
Why must a proposed payment or file write require more than a model's permission? The executor needs independent authorization, exact-action user approval and audit/rollback controls. Add a maximum step count before extending this into a loop.